In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from math import comb

import matplotlib.pyplot as plt
import numpy as np

from eval import load

FIGURE_DIR = Path("../outputs/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

LOAD_CONFIGS = {
    "mmlu": {"dataset_size": 1000, "numvariants": 30},
}


def _ensure_legacy_columns(df):
    df = df.copy()
    if "passed" not in df.columns and "Correct" in df.columns:
        df["passed"] = df["Correct"].astype(int)
    if "Correct" not in df.columns and "passed" in df.columns:
        df["Correct"] = df["passed"].astype(int)
    return df


def load_mmlu_retok(model_name):
    return _ensure_legacy_columns(load.load_mmlu(model_name, variant_type="retok", **LOAD_CONFIGS["mmlu"]))


def calculate_pass_k(n_total, num_correct, k):
    incorrect = n_total - num_correct
    if incorrect < k:
        return 1.0
    return 1.0 - comb(incorrect, k) / comb(n_total, k)


In [ ]:
model_name = "allenai/OLMo-2-1124-7B-Instruct"
df_data = load_mmlu_retok(model_name)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5), constrained_layout=True)

dataset = "mmlu"
n_total_tasks = df_data.prompti.nunique()
n_samples = int(df_data.prompti.value_counts().iloc[0])
ks = np.arange(1, min(n_samples, 150), 2)

canonical_counts = (df_data[df_data.p == 0].answer_prob.values * n_samples).astype(int)
passk_canon = np.array([
    [calculate_pass_k(n_samples, canonical_counts[task_id], k) for k in ks]
    for task_id in range(n_total_tasks)
])
ax.errorbar(ks, y=np.mean(passk_canon, axis=0), color="forestgreen", label="pass@k", capsize=4, linestyle="-")

retok_counts = df_data.groupby("prompti").Correct.sum().values
passk_retok = np.array([
    [calculate_pass_k(n_samples, retok_counts[task_id], k) for k in ks]
    for task_id in range(n_total_tasks)
])
ax.errorbar(ks, y=np.mean(passk_retok, axis=0), color="forestgreen", label="pass@retok", capsize=4, linestyle="--")

passk_random = np.array([
    [calculate_pass_k(n_samples, int(n_samples * 0.25), k) for k in ks]
    for _ in range(n_total_tasks)
])
ax.errorbar(ks, y=np.mean(passk_random, axis=0), color="gray", label="pass@k random", capsize=4, linestyle="-")

ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xlabel("k")
ax.set_ylabel("Pass rate")
ax.set_xlim(0, 50)
fig.savefig(FIGURE_DIR / "passk_mmlu_random.svg", bbox_inches="tight")
